In [1]:
#!pip install -U ultralytics
import ultralytics
ultralytics.checks()  # Verify setup

Ultralytics 8.3.229 🚀 Python-3.9.5 torch-1.12.1+cu113 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
Setup complete ✅ (12 CPUs, 7.4 GB RAM, 106.6/1006.9 GB disk)


In [2]:
import os
import json
from tqdm import tqdm

import json
from collections import defaultdict
from ultralytics import YOLO

# Inferencing

In [ ]:
DATASET_ROOT = "/src/data/test/samples"
OUTPUT_FILE = "/result/submission.json"
MODEL_WEIGHTS = "/src/weight/best.pt"

CONFIDENCE_THRESHOLD = 0.25

# Inference with tracker

In [ ]:


def inference():
    # 1. Load the model
    model = YOLO(MODEL_WEIGHTS)

    # List to hold the final data for all videos
    all_videos_output = []

    # 2. Find all video folders (sorted ensures consistent order)
    # We look for folders in dataset/samples/
    if not os.path.exists(DATASET_ROOT):
        print(f"Error: Dataset root '{DATASET_ROOT}' not found.")
        return

    video_folders = sorted(os.listdir(DATASET_ROOT))

    print(f"Found {len(video_folders)} folders. Starting processing...")

    # 3. Iterate through each folder
    for folder_name in tqdm(video_folders):
        video_path = os.path.join(DATASET_ROOT, folder_name, "drone_video.mp4")

        # Skip if the video file doesn't exist in this folder
        if not os.path.exists(video_path):
            continue

        # The folder name IS the video_id (e.g., "drone_video_001")
        video_id = folder_name

        # --- TRACKING LOGIC ---
        # Run tracker with stream=True to manage memory
        results = model.track(source=video_path, conf=0.3, iou=0.5, stream=True, verbose=False)

        # Dictionary to group results by Track ID
        # Structure: { track_id: [ {frame, x1...}, {frame, x1...} ] }
        track_history = defaultdict(list)

        for frame_index, r in enumerate(results):
            # Check if we have detected objects with Track IDs
            if r.boxes and r.boxes.id is not None:
                boxes = r.boxes.xyxy.int().cpu().tolist()
                track_ids = r.boxes.id.int().cpu().tolist()

                for track_id, box in zip(track_ids, boxes):
                    x1, y1, x2, y2 = box

                    bbox_entry = {
                        "frame": frame_index,
                        "x1": x1,
                        "y1": y1,
                        "x2": x2,
                        "y2": y2
                    }
                    track_history[track_id].append(bbox_entry)

        # --- FORMATTING SINGLE VIDEO RESULT ---
        # Convert the history dict into the list format required by the schema
        detections_list = []
        for track_id, bboxes in track_history.items():
            detections_list.append({
                "bboxes": bboxes
            })

        # Append this video's data to the master list
        all_videos_output.append({
            "video_id": video_id,
            "detections": detections_list
        })

    # 4. Save the final merged JSON
    with open(OUTPUT_FILE, "w") as f:
        json.dump(all_videos_output, f, indent=3)

    print(f"\nProcessing complete! Results saved to {OUTPUT_FILE}")

inference()

Found 6 folders. Starting processing...


  0%|                                                                                                                              | 0/6 [00:00<?, ?it/s]

requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.7 MB ? eta -:--:--
   ━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.3/1.7 MB ? eta -:--:--
   ━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/1.7 MB 1.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━ 1.0/1.7 MB 2.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━ 1.3/1.7 MB 2.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.8 MB/s  0:00:00

requirements: AutoUpdate success ✅ 3.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



 17%|███████████████████▌                                                                                                 | 1/6 [04:23<21:56, 263.39s/it]